# Lab 1D: 1:N Multi-Project Deployment

Deploy **multiple team projects** under a **single shared AI Foundry Account** -- the 1:N pattern.

## What Gets Deployed

| Resource | Purpose |
|----------|--------|
| AI Foundry Account (1) | Shared departmental account |
| Foundry Projects (N) | One workspace per team (alpha, beta, gamma) |
| APIM Connections (N) | Each project gets its own gateway connection to landing zone models |

## Why 1:N?

In enterprise environments, teams within the same department share infrastructure costs while maintaining **project-level isolation**:

- **Cost efficiency** -- one AI account instead of N accounts
- **Isolation** -- each team has its own project workspace, agents, and data
- **Shared RBAC** -- account-level roles propagate; project-level roles scope access
- **Unified management** -- single account to monitor, patch, and govern

> Prerequisite: Complete **Lab 1A** first to deploy the Landing Zone

In [44]:
# Enable automatic masking of Azure resource names in print output
import sys
sys.path.insert(0, "../..")
from secure_print import install
install()

secure_print: Azure resource masking enabled


## Step 1: Load Landing Zone Configuration

In [45]:
import os
from pathlib import Path

env_file = Path("../../.env")
with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

AI_ENDPOINT = os.environ['AI_ENDPOINT']
APIM_URL = os.environ['APIM_URL']
APIM_KEY = os.environ['APIM_KEY']
MODEL_NAME = os.environ['MODEL_NAME']

print(f"Landing Zone AI Endpoint: {AI_ENDPOINT}")
print(f"Landing Zone APIM URL:    {APIM_URL}")
print(f"APIM Key:                 {APIM_KEY[:2]}... (hidden)")
print(f"Model Name:               {MODEL_NAME}")

Landing Zone AI Endpoint: https://fou***.cognitiveservices.azure.com/
Landing Zone APIM URL:    https://fou***.azure-api.net/openai
APIM Key:                 53... (hidden)
Model Name:               gpt-4.1-mini


## Step 2: Set Variables

In [46]:
MULTI_RG = "lab1d-foundry-multi-project"
LOCATION = "eastus2"
TEAM_NAMES = ["alpha", "beta", "gamma"]
PROJECT_COUNT = len(TEAM_NAMES)

print(f"Resource Group: {MULTI_RG}")
print(f"Teams:          {', '.join(TEAM_NAMES)}")
print(f"Project Count:  {PROJECT_COUNT}")

Resource Group: lab1d-foundry-multi-project
Teams:          alpha, beta, gamma
Project Count:  3


## Step 3: Create Resource Group

In [47]:
!az group create -n "{MULTI_RG}" -l "{LOCATION}" -o table

Location    Name
----------  ---------------------------
eastus2     lab1d-foundry-multi-project


## Step 4: Deploy 1:N Infrastructure

Deploys one AI Foundry Account with N projects and their APIM connections.

Takes ~2-3 minutes

In [48]:
import subprocess, json

PRINCIPAL_ID = subprocess.run(
    'az ad signed-in-user show --query id -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()

team_names_json = json.dumps(TEAM_NAMES)

# Use subprocess.run to avoid shell quoting issues with JSON arrays
result = subprocess.run(
    [
        "az", "deployment", "group", "create",
        "-g", MULTI_RG,
        "--template-file", "main.bicep",
        "-p", f"deployerPrincipalId={PRINCIPAL_ID}",
        "-p", f"apimUrl={APIM_URL}",
        "-p", f"modelName={MODEL_NAME}",
        "-p", f"apimSubscriptionKey={APIM_KEY}",
        "-p", f"projectCount={PROJECT_COUNT}",
        "-p", f"teamNames={team_names_json}",
        "-o", "table",
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Name    State      Timestamp                         Mode         ResourceGroup
------  ---------  --------------------------------  -----------  ---------------------------
main    Succeeded  2026-02-06T12:23:53.764095+00:00  Incremental  lab1d-foundry-multi-project



## Step 5: Get Outputs

In [49]:
import subprocess, json
from pathlib import Path

r = subprocess.run(
    f'az deployment group show -g "{MULTI_RG}" -n main --query properties.outputs -o json',
    shell=True, capture_output=True, text=True
)

if r.returncode != 0 or not r.stdout.strip():
    raise RuntimeError(
        f"Failed to retrieve deployment outputs. "
        f"Make sure the deployment in Step 4 succeeded.\n{r.stderr}"
    )

out = json.loads(r.stdout)

ACCOUNT_NAME = out['accountName']['value']
ACCOUNT_ENDPOINT = out['accountEndpoint']['value']
PROJECT_NAMES = out['projectNames']['value']
PROJECT_ENDPOINTS = out['projectEndpoints']['value']
APIM_CONNECTIONS = out['apimConnectionNames']['value']

print(f"Shared Account:    {ACCOUNT_NAME}")
print(f"Account Endpoint:  {ACCOUNT_ENDPOINT}")
print()
for i, (name, endpoint) in enumerate(zip(PROJECT_NAMES, PROJECT_ENDPOINTS)):
    print(f"  Project {i+1} ({TEAM_NAMES[i]}): {name}")
    print(f"    Endpoint:   {endpoint}")
    print(f"    Connection: {APIM_CONNECTIONS[i]}")

# Append outputs to .env file
env_file = Path("../../.env")
with open(env_file, 'a') as f:
    f.write(f"\n# Lab 1D: Multi-Project Outputs\n")
    f.write(f"MULTI_ACCOUNT={ACCOUNT_NAME}\n")
    f.write(f"MULTI_ACCOUNT_ENDPOINT={ACCOUNT_ENDPOINT}\n")
    for i, name in enumerate(PROJECT_NAMES):
        team = TEAM_NAMES[i].upper()
        f.write(f"PROJECT_{team}={name}\n")
        f.write(f"PROJECT_{team}_ENDPOINT={PROJECT_ENDPOINTS[i]}\n")
        f.write(f"PROJECT_{team}_CONNECTION={APIM_CONNECTIONS[i]}\n")
print(f"\nOutputs appended to {env_file}")

Shared Account:    foundry-multi-ycxlbw
Account Endpoint:  https://fou***.cognitiveservices.azure.com/

  Project 1 (alpha): project-alpha-ycxlbw
    Endpoint:   https://fou***.services.ai.azure.com/api/projects/project-alpha-ycxlbw
    Connection: landing-zone-apim-alpha
  Project 2 (beta): project-beta-ycxlbw
    Endpoint:   https://fou***.services.ai.azure.com/api/projects/project-beta-ycxlbw
    Connection: landing-zone-apim-beta
  Project 3 (gamma): project-gamma-ycxlbw
    Endpoint:   https://fou***.services.ai.azure.com/api/projects/project-gamma-ycxlbw
    Connection: landing-zone-apim-gamma

Outputs appended to ../../.env


## Step 6: Test Each Project

Connect to every project independently and verify each can reach the landing zone models through its own APIM connection.

> APIM gateway connections require the **Agent + Responses API** path; `chat.completions` is not supported for this connection type.

In [50]:
!pip install azure-ai-projects==2.0.0b2 azure-identity agent-framework-azure-ai==1.0.0b251223 -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [51]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

credential = DefaultAzureCredential()

print(f"Testing {len(PROJECT_ENDPOINTS)} projects...")
print("=" * 60)

for i, endpoint in enumerate(PROJECT_ENDPOINTS):
    team = TEAM_NAMES[i]
    project_name = PROJECT_NAMES[i]
    gateway_model = f"{APIM_CONNECTIONS[i]}/{MODEL_NAME}"

    client = AIProjectClient(credential=credential, endpoint=endpoint)

    # APIM connections require the Agent + Responses API (chat.completions won't work)
    agent = client.agents.create_version(
        agent_name=f"verify-{team}",
        definition=PromptAgentDefinition(
            model=gateway_model,
            instructions="Reply in one sentence only.",
        ),
    )

    openai_client = client.get_openai_client()
    response = openai_client.responses.create(
        input=f"Reply in one sentence: which team are you serving? Team {team}.",
        extra_body={
            "agent": {
                "name": agent.name,
                "version": agent.version,
                "type": "agent_reference",
            }
        },
    )
    answer = response.output_text if hasattr(response, "output_text") else str(response.output)

    # Cleanup verification agent
    client.agents.delete(agent_name=agent.name)

    print(f"\nProject {i+1} ({team}): {project_name}")
    print(f"  Gateway model: {gateway_model}")
    print(f"  Response: {answer}")

print("\n" + "=" * 60)
print(f"All {len(PROJECT_ENDPOINTS)} projects verified.")

Testing 3 projects...



Project 1 (alpha): project-alpha-ycxlbw
  Gateway model: landing-zone-apim-alpha/gpt-4.1-mini
  Response: I am serving Team alpha.

Project 2 (beta): project-beta-ycxlbw
  Gateway model: landing-zone-apim-beta/gpt-4.1-mini
  Response: I am here to serve Team Beta!

Project 3 (gamma): project-gamma-ycxlbw
  Gateway model: landing-zone-apim-gamma/gpt-4.1-mini
  Response: I am serving Team Gamma with full support and dedication.

All 3 projects verified.


## Step 7: Create Per-Team Agents

Each project gets its own agent -- demonstrating that agents are scoped to projects, not accounts.

In [52]:
from azure.ai.projects.models import PromptAgentDefinition

agents = []

for i, endpoint in enumerate(PROJECT_ENDPOINTS):
    team = TEAM_NAMES[i]
    gateway_model = f"{APIM_CONNECTIONS[i]}/{MODEL_NAME}"
    client = AIProjectClient(credential=credential, endpoint=endpoint)

    agent = client.agents.create_version(
        agent_name=f"team-{team}-agent",
        definition=PromptAgentDefinition(
            model=gateway_model,
            instructions=(
                f"You are the dedicated assistant for Team {team.capitalize()}. "
                f"Always identify yourself as the Team {team.capitalize()} assistant. "
                "Keep responses brief."
            ),
        ),
    )
    agents.append((team, agent, client))
    print(f"Agent created: {agent.name} v{agent.version} (project: {PROJECT_NAMES[i]})")

print(f"\n{len(agents)} team agents created across {len(PROJECT_ENDPOINTS)} projects.")

Agent created: team-alpha-agent v1 (project: project-alpha-ycxlbw)
Agent created: team-beta-agent v1 (project: project-beta-ycxlbw)
Agent created: team-gamma-agent v1 (project: project-gamma-ycxlbw)

3 team agents created across 3 projects.


In [53]:
# Invoke each agent
for team, agent, client in agents:
    openai_client = client.get_openai_client()
    response = openai_client.responses.create(
        input=f"Hello! Which team do you belong to?",
        extra_body={
            "agent": {
                "name": agent.name,
                "version": agent.version,
                "type": "agent_reference"
            }
        }
    )
    answer = response.output_text if hasattr(response, 'output_text') else str(response.output)
    print(f"Team {team}: {answer}")
    print()

Team alpha: Hello! I am the assistant for Team Alpha. How can I help you today?

Team beta: Hello! I am the assistant for Team Beta. How can I help you today?

Team gamma: Hello! I am the assistant for Team Gamma. How can I help you today?



## Done!

You deployed the **1:N pattern**: one AI Foundry Account hosting multiple team projects.

### Key Concepts

| Concept | Description |
|---------|-------------|
| 1:N Pattern | One AI Account, many projects -- shared infra with project-level isolation |
| Bicep loop | `for i in range(0, projectCount)` creates N projects and connections |
| Agent scoping | Agents live inside a project; Team Alpha cannot see Team Beta's agents |
| Shared RBAC | Account-level `Cognitive Services User` role applies to all projects |
| Per-project connections | Each project has its own uniquely-named APIM connection (e.g., `landing-zone-apim-alpha`) |

### When to use 1:N vs. separate accounts

| Scenario | Recommendation |
|----------|----------------|
| Teams in the same department / cost center | 1:N (this lab) |
| Teams with different compliance boundaries | Separate accounts (Lab 1B) |
| Dev / staging / prod environments | Separate accounts per environment |
| Rapid prototyping with many small teams | 1:N to avoid account sprawl |

## Cleanup (Optional)

In [54]:
# for team, agent, client in agents:
#     client.agents.delete(agent_name=agent.name)
#     print(f"Deleted agent: {agent.name}")
# !az group delete -n "{MULTI_RG}" --yes --no-wait